# 4-6 全部つないだ完成版

4-2 〜 4-5 で「読込 → 結合 → 集計 → グラフ → Excel 書出」が一通りできました。最後の仕上げは:

**全部を 1 つの関数にまとめて、来月もセル 1 つ実行で済む形にする**

これが本講座の **最終ゴール** です。1-5 で習った **関数 `def`** が、ここで真価を発揮します。

## このノートのゴール

**`generate_monthly_report(target_month)`** という関数を作る。

```python
generate_monthly_report("2026-01")
# → output/report_2026-01.xlsx と output/charts/*.png が生成される
```

**この 1 行が、毎月の月次レポート作業のすべて** になります。

- 引数 1 つで対象月を切り替えられる
- フォルダ・ファイル命名は対象月に応じて自動
- 来月分のファイルが届いたら、引数を `"2026-02"` に変えるだけ

## 「関数化」の何が嬉しいか

ここまで 4-2 → 4-3 → 4-4 → 4-5 と書いてきたコードは、**そのままだと「2026 年 1 月用のスクリプト」** です。

2 月になったら:

- `sales_2026-01` を `sales_2026-02` に書き換え
- `report_2026-01.xlsx` を `report_2026-02.xlsx` に書き換え
- タイトルの「2026年1月」を「2026年2月」に書き換え
- …他、似たような書き換えを 30 箇所くらい

**書き換え漏れが必ず起きる**。これを **対象月を引数で受け取る関数** にまとめてしまえば、間違いようがない。

## 準備

In [ ]:
!pip install -q japanize-matplotlib

# === Colab ===
# from google.colab import drive
# drive.mount('/content/drive')
# BASE = "/content/drive/MyDrive/python-data-basics"

# === ローカル ===
BASE = "."

import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import japanize_matplotlib
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage

print("準備 OK")

## 小さな関数に分けて作る

いきなり 200 行の関数を書こうとすると、デバッグが大変。**4-2 〜 4-5 の各工程を、それぞれ小さな関数** にしておくと、後で組み立てが楽です。

1-5 で「**関数は、入力と出力をはっきりさせると使い回しやすい**」と言いました。それを実践します。

### 1) ファイル読み込み (4-2 の中身)

In [ ]:
def load_all_sales(sales_dir: str) -> pd.DataFrame:
    """sales_dir 配下の Excel を全部読み、prefecture_code 列を付けて 1 つの DataFrame に"""
    files = sorted(Path(sales_dir).glob("sales_*.xlsx"))
    dfs = []
    for f in files:
        d = pd.read_excel(f, sheet_name="売上")
        d["prefecture_code"] = int(f.stem.split("_")[2])
        dfs.append(d)
    return pd.concat(dfs, ignore_index=True)

### 2) マスタ結合 (4-3 の前半)

In [ ]:
def join_masters(all_sales: pd.DataFrame, data_dir: str) -> pd.DataFrame:
    """商品マスタ・支店マスタを結合し、profit 列も計算"""
    mp = pd.read_excel(f"{data_dir}/master_products.xlsx")
    mb = pd.read_excel(f"{data_dir}/master_branches.xlsx")
    df = (
        all_sales
        .merge(mp[["product_code", "category", "cost"]], on="product_code", how="left")
        .merge(mb, on="prefecture_code", how="left")
    )
    df["profit"] = df["amount"] - df["cost"] * df["quantity"]
    return df

### 3) 集計 (4-3 の後半)

In [ ]:
def aggregate(df: pd.DataFrame) -> dict:
    """全集計を辞書で返す。後の工程で何度も参照する用"""
    return {
        "by_date":     df.groupby("date")["amount"].sum(),
        "by_region":   df.groupby("region").agg(売上=("amount","sum"), 利益=("profit","sum"), 件数=("amount","count")).sort_values("売上", ascending=False),
        "by_pref":     df.groupby("prefecture_name").agg(売上=("amount","sum"), 利益=("profit","sum"), 件数=("amount","count")).sort_values("売上", ascending=False),
        "by_product":  df.groupby(["product_code","product_name","category"]).agg(売上=("amount","sum"), 利益=("profit","sum"), 数量=("quantity","sum")).sort_values("売上", ascending=False),
        "pivot_region_cat": df.pivot_table(index="region", columns="category", values="amount", aggfunc="sum", fill_value=0),
    }

### 4) グラフ描画 (4-4 の中身)

**毎回同じ体裁** なので、グラフを 1 枚描く処理 (`save_bar_chart` 等) を共通の関数にしてしまいます。

In [ ]:
def save_charts(agg: dict, charts_dir: str, target_month: str) -> None:
    """集計結果から 5 枚のグラフを charts_dir に保存"""
    os.makedirs(charts_dir, exist_ok=True)
    fmt_man = ticker.FuncFormatter(lambda x, _: f"{x/10000:,.0f}")

    # 1) 日次推移
    ax = agg["by_date"].plot(figsize=(12, 4), marker="o", color="steelblue")
    ax.yaxis.set_major_formatter(fmt_man)
    plt.title(f"全社 日次売上推移 ({target_month})", fontsize=14, fontweight="bold")
    ax.text(0, 1.02, "(万円)", transform=ax.transAxes, ha="left", fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{charts_dir}/daily_sales.png", dpi=150, bbox_inches="tight"); plt.close()

    # 2) 地域別
    ax = agg["by_region"]["売上"].plot.bar(figsize=(9, 4), color="steelblue")
    ax.yaxis.set_major_formatter(fmt_man)
    plt.title("地域ブロック別 売上", fontsize=14, fontweight="bold")
    ax.text(0, 1.02, "(万円)", transform=ax.transAxes, ha="left", fontsize=10)
    plt.xticks(rotation=0); plt.tight_layout()
    plt.savefig(f"{charts_dir}/by_region.png", dpi=150, bbox_inches="tight"); plt.close()

    # 3) 県別 TOP10 横棒
    top10 = agg["by_pref"]["売上"].head(10)
    ax = top10[::-1].plot.barh(figsize=(8, 5), color="steelblue")
    ax.xaxis.set_major_formatter(fmt_man)
    plt.title("県別 売上 TOP10", fontsize=14, fontweight="bold")
    ax.text(0, 1.02, "(万円)", transform=ax.transAxes, ha="left", fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{charts_dir}/by_pref_top10.png", dpi=150, bbox_inches="tight"); plt.close()

    # 4) 商品別
    by_p = agg["by_product"].reset_index().set_index("product_name")["売上"]
    ax = by_p.plot.bar(figsize=(9, 4), color="steelblue")
    ax.yaxis.set_major_formatter(fmt_man)
    plt.title("商品別 売上", fontsize=14, fontweight="bold")
    ax.text(0, 1.02, "(万円)", transform=ax.transAxes, ha="left", fontsize=10)
    plt.xticks(rotation=20); plt.tight_layout()
    plt.savefig(f"{charts_dir}/by_product.png", dpi=150, bbox_inches="tight"); plt.close()

    # 5) 地域 × カテゴリ 積み上げ
    ax = agg["pivot_region_cat"].plot.bar(stacked=True, figsize=(10, 5), colormap="Pastel1")
    ax.yaxis.set_major_formatter(fmt_man)
    plt.title("地域 × カテゴリ 売上内訳", fontsize=14, fontweight="bold")
    ax.text(0, 1.02, "(万円)", transform=ax.transAxes, ha="left", fontsize=10)
    plt.xticks(rotation=0)
    plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.savefig(f"{charts_dir}/region_category_stacked.png", dpi=150, bbox_inches="tight"); plt.close()

> 💡 `plt.close()` を毎回呼んでいるのは、**関数で大量にグラフを描くと Jupyter 側にゴミが残る** ため。`plt.show()` の代わりに `close` でクリーンに片付ける。

### 5) Excel レポート書き出し (4-5 の中身)

In [ ]:
def write_report(df: pd.DataFrame, agg: dict, report_path: str, charts_dir: str, target_month: str) -> None:
    """レポート Excel を書き出し、概要シートにグラフ画像を貼り込む"""
    summary = pd.DataFrame({
        "項目": ["対象月", "営業日数", "取引件数", "売上合計 (円)", "利益合計 (円)", "平均日商 (円)"],
        "値": [
            target_month,
            df["date"].nunique(),
            len(df),
            int(df["amount"].sum()),
            int(df["profit"].sum()),
            int(df["amount"].sum() / df["date"].nunique()),
        ],
    })

    # 表を 5 シートに書く
    with pd.ExcelWriter(report_path, engine="openpyxl") as w:
        summary.to_excel(w, sheet_name="概要", index=False)
        agg["by_region"].to_excel(w, sheet_name="地域別")
        agg["by_pref"].to_excel(w, sheet_name="都道府県別")
        agg["by_product"].to_excel(w, sheet_name="商品別")
        df.to_excel(w, sheet_name="明細", index=False)

    # 概要シートに画像を貼る + 列幅調整
    wb = load_workbook(report_path)
    ws = wb["概要"]
    ws.add_image(XLImage(f"{charts_dir}/daily_sales.png"),   "D2")
    ws.add_image(XLImage(f"{charts_dir}/by_region.png"),     "D24")
    ws.add_image(XLImage(f"{charts_dir}/by_pref_top10.png"), "D48")
    for col, width in zip("ABCDEFGHIJKL", [12, 14, 18, 8, 10, 12, 14, 14, 10, 16, 12, 14]):
        wb["明細"].column_dimensions[col].width = width
    wb.save(report_path)

### 6) 全部をつなぐ司令塔の関数

**これが本ノートの主役** です。引数は **対象月の文字列 1 つ** のみ。中で上の関数を順番に呼ぶだけ。

In [ ]:
def generate_monthly_report(target_month: str, base: str = ".") -> str:
    """target_month (例 '2026-01') のレポート Excel を生成し、出力パスを返す"""
    data_dir = f"{base}/data/capstone"
    sales_dir = f"{data_dir}/sales_{target_month}"
    out_dir = f"{base}/output"
    charts_dir = f"{out_dir}/charts"
    report_path = f"{out_dir}/report_{target_month}.xlsx"

    os.makedirs(out_dir, exist_ok=True)

    print(f"[1/4] {sales_dir} から読み込み...")
    all_sales = load_all_sales(sales_dir)
    print(f"      → {len(all_sales):,} 行")

    print("[2/4] マスタ結合 + 利益計算...")
    df = join_masters(all_sales, data_dir)

    print("[3/4] 集計 + グラフ作成...")
    agg = aggregate(df)
    save_charts(agg, charts_dir, target_month)

    print("[4/4] Excel レポート書き出し...")
    write_report(df, agg, report_path, charts_dir, target_month)

    print(f"\n完了 → {report_path}")
    return report_path

## いざ実行 — セル 1 つで完成

In [ ]:
generate_monthly_report("2026-01", base=BASE)

**`output/report_2026-01.xlsx`** が出来上がっているはずです。Excel で開いて中身を確認してみてください。

そして来月 (2026年2月) になって、`data/capstone/sales_2026-02/` に新しい 47 ファイルが届いたら:

```python
generate_monthly_report("2026-02", base=BASE)
```

**これだけ**。引数を変えるだけで全部やってくれます。

## 練習問題

本文の関数を少しだけ変える形の演習です。

1. **`generate_monthly_report("2026-01")` をもう一度実行** してみてください。何度実行しても **同じ結果** が得られる (=冪等) ことを確認します。これが「自動化スクリプト」の最低条件。
2. **対象月を関数の中で表示** したい。`generate_monthly_report` の **`[4/4]` のあと** に、`print(f"対象月: {target_month} のレポートが完成しました")` を 1 行追加してください。
3. **`save_charts` の `colormap`** を `"Pastel1"` から **`"Set2"`** に変えて、もう一度実行してみてください。Excel に貼り込まれるグラフの配色が変わるはず。

In [ ]:
# ここに書いてみてください (本文の関数定義を再実行してから呼ぶ)


<details>
<summary>答えを見る</summary>

```python
# 1. もう一度実行 — 同じ Excel が上書きされるだけ
generate_monthly_report("2026-01", base=BASE)

# 2. print を 1 行追加した版 (generate_monthly_report の末尾に)
# print(f"対象月: {target_month} のレポートが完成しました")

# 3. save_charts 内の colormap を "Set2" に変えて、上のセルから再定義→実行
```

</details>

## 設計のポイント

今回作った関数群の **構造** を改めて見てみます:

```
generate_monthly_report(target_month)   ← 司令塔
  ├─ load_all_sales(sales_dir)         ← 1 工程目
  ├─ join_masters(all_sales, data_dir)  ← 2 工程目
  ├─ aggregate(df)                       ← 3 工程目
  ├─ save_charts(agg, charts_dir, …)    ← 4 工程目
  └─ write_report(df, agg, …)            ← 5 工程目
```

**「司令塔関数」+ 「工程ごとの関数」** という構造は、**業務の自動化スクリプトの王道** です。

メリット:

- **1 工程ずつ単体テストできる** — `load_all_sales(...)` だけ呼んで結果を確かめられる
- **どこで失敗したか分かりやすい** — `[3/4]` で止まったら集計周りの問題
- **来月の改修が局所的** — 「集計に新しい指標を増やしたい」なら `aggregate` だけ触る
- **他のレポートにも転用できる** — `save_charts` を別シナリオで使い回す等

## まとめ

- 4-2 〜 4-5 の処理を **小さな関数 5 つ + 司令塔関数 1 つ** に分割
- 司令塔 **`generate_monthly_report("YYYY-MM")`** を 1 行呼ぶだけでレポート完成
- 来月以降の運用は **引数の対象月を変えるだけ**
- 「**業務ロジックは関数に閉じ込め、呼ぶ側は引数だけ**」が自動化スクリプトの王道

ここまでで **「事務職の月次レポート作業」を Python で完全自動化** することができました。

次の **4-7** では、いよいよ **VBA で同じ処理を書いたものと、所要時間を比較** します。「ファイルを開かない pandas」が VBA に対してどれだけ速いか、実測で見ていきましょう。